# Лабораторная работа по теме: «Метод главных компонент»

### Задание

Методом главных компонент построить два новых признака для описания объектов из датасета задания 2 и кластеризовать данные по этим двум признакам на два кластера (любым методом). Сравнить качество кластеризации  из задания 2 и из этого задания, используя функционалы качества из дополнительных материалов ИТМО (смотреть только внутренние оценки), дать интерпретацию для полученных кластеров и сделать общий вывод по работе. В этом задании можно использовать готовые функции РСА.

### Подключение необходимых библиотек и загрузка датасета данных

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import plotly.express as px

df = pd.read_csv('/adult.csv')
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Предобработка данных

In [32]:
columns_for_clustering = [
    'age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week',
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

df = df[columns_for_clustering + ['salary']]
df = df.replace(' ?', np.nan).dropna().reset_index(drop=True)

print(f"Размер данных после очистки: {df.shape}")

# Кодирование категориальных признаков
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation',
                    'relationship', 'race', 'sex', 'native-country']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

X = df[columns_for_clustering].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Предобработка данных завершена")

Размер данных после очистки: (32561, 14)
Предобработка данных завершена


### Применение PCA

In [33]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Доля объяснённой дисперсии")
print(f"\tPC1: {pca.explained_variance_ratio_[0]:.4f} ({pca.explained_variance_ratio_[0]*100:.2f}%)")
print(f"\tPC2: {pca.explained_variance_ratio_[1]:.4f} ({pca.explained_variance_ratio_[1]*100:.2f}%)")
print(f"\tСуммарно: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_)*100:.2f}%)")

# DataFrame с новыми признаками
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca.head()

Доля объяснённой дисперсии
	PC1: 0.1629 (16.29%)
	PC2: 0.1081 (10.81%)
	Суммарно: 0.2710 (27.10%)


,PC1,PC2
0,-0.684522,0.394423
1,-0.773449,-0.104576
2,-0.954585,-0.465678
3,-0.363918,-2.808264
4,2.050872,0.851670


### Вклады признаков в компоненты

In [34]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=columns_for_clustering
)

print("Наиболее важные признаки для каждой компоненты")
print("\tPC1:")
print(loadings['PC1'].abs().sort_values(ascending=False).round(3))
print("\n\tPC2:")
print(loadings['PC2'].abs().sort_values(ascending=False).round(3))

Наиболее важные признаки для каждой компоненты
	PC1:
relationship      0.523
sex               0.462
hours-per-week    0.377
marital-status    0.322
age               0.283
education-num     0.217
workclass         0.211
occupation        0.166
race              0.157
capital-gain      0.141
capital-loss      0.111
education         0.101
native-country    0.036
Name: PC1, dtype: float64

	PC2:
education-num     0.616
education         0.609
sex               0.279
relationship      0.228
native-country    0.172
capital-gain      0.158
age               0.152
occupation        0.135
workclass         0.112
hours-per-week    0.069
capital-loss      0.063
marital-status    0.042
race              0.015
Name: PC2, dtype: float64


### Кластеризация на PCA-признаках

In [35]:
kmeans_pca = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca)

df_pca['Cluster'] = labels_pca
df['Cluster_PCA'] = labels_pca

print("\tРаспределение по кластерам:")
print(df_pca['Cluster'].value_counts())

	Распределение по кластерам:
Cluster
1    18586
0    13975
Name: count, dtype: int64


### Визуализация кластеров

In [46]:
fig = px.scatter(
    df_pca, x='PC1', y='PC2',
    color='Cluster',
    title='<b>Кластеры в пространстве двух главных компонент (PCA)</b>',
    opacity=0.7,
    color_discrete_map={'0': 'blue', '1': 'red'}
)
fig.update_layout(
    width=1000,
    height=700,
    plot_bgcolor='white',
    title={
        'text': '<b>Кластеры в пространстве двух главных компонент (PCA)</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'color': 'black', 'size': 16}
    },
    font={'color': 'black'}
)
fig.show()

### Оценка качества кластеризации (внутренние метрики)

In [39]:
def evaluate_clustering_full(X, labels, method_name):
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)
    print(f"\n{method_name}:")
    print(f"  Silhouette Score:      {sil:.4f}")
    print(f"  Calinski-Harabasz:     {ch:.2f}")
    print(f"  Davies-Bouldin Index:  {db:.4f}")
    return {'silhouette': sil, 'calinski_harabasz': ch, 'davies_bouldin': db}

# Метрики в пространстве PCA
metrics_pca_2d = evaluate_clustering_full(X_pca, labels_pca, "KMeans на PCA (2D)")
metrics_pca_original = evaluate_clustering_full(X_scaled, labels_pca, "Сравнение на исходных 14 признаках:\nKMeans после PCA (оценка на 14D)")


KMeans на PCA (2D):
  Silhouette Score:      0.4421
  Calinski-Harabasz:     26446.29
  Davies-Bouldin Index:  0.9140

Сравнение на исходных 14 признаках:
KMeans после PCA (оценка на 14D):
  Silhouette Score:      0.1477
  Calinski-Harabasz:     4569.67
  Davies-Bouldin Index:  2.4327


### Сравнительный анализ метрик

Анализ внутренних метрик качества кластеризации позволяет сделать следующие выводы:

- Коэффициент силуэта. Наивысшее значение зафиксировано у агломеративной кластеризации из задания 2 (0.3067), что говорит о лучшей отделимости кластеров. Силуэт KMeans после PCA, рассчитанный на исходных 14 признаках (0.1477), практически идентичен результату обычного KMeans из задания 2 (0.1494). Значение в двумерном пространстве PCA (0.4421) значительно выше, но это ожидаемый эффект снижения размерности.

- Индекс Калински-Харабаса. Максимальное значение в задании 2 показал KMeans (4595.93). После применения PCA данный индекс составил 4569.67 (на 14 признаках), что также очень близко к результату KMeans из предыдущего задания и свидетельствует о сохранении компактности кластеров.

- Индекс Дэвиса-Болдина. В пространстве PCA значение составило 0.9140. Однако при оценке на исходных 14 признаках индекс вырос до 2.4327, что указывает на умеренное пересечение кластеров и является самым слабым показателем среди рассмотренных.

### Интерпретация полученных кластеров (PCA-признаки)

Для интерпретации проанализируем средние значения исходных признаков в каждом кластере.

In [40]:
print("Средние значения исходных признаков в кластерах:")
print(df.groupby('Cluster_PCA')[columns_for_clustering].mean().round(2))

print("\nРаспределение salary по кластерам (%):")
print(pd.crosstab(df['Cluster_PCA'], df['salary'], normalize='index').round(3))

Средние значения исходных признаков в кластерах:
               age  education-num  capital-gain  capital-loss  hours-per-week  \
Cluster_PCA                                                                     
0            32.87           9.71        340.02         45.06           35.00   
1            42.88          10.36       1632.28        119.07           44.53   

             workclass  education  marital-status  occupation  relationship  \
Cluster_PCA                                                                   
0                 3.58      10.26            3.39        5.94          2.85   
1                 4.08      10.33            2.02        7.05          0.39   

             race   sex  native-country  
Cluster_PCA                              
0            3.51  0.31           36.63  
1            3.78  0.94           36.78  

Распределение salary по кластерам (%):
salary       <=50K   >50K
Cluster_PCA              
0            0.922  0.078
1            0.637  0.3

**Кластер 0** (13 975 человек — 42.9% выборки):  
- Средний возраст: 32.87 года  
- Уровень образования: 9.71 лет  
- Среднее количество рабочих часов в неделю: 35.00
- Капитальный доход и потери — низкие  
- Доля людей с доходом >50K: 7.8%

**Кластер 1** (18 586 человек — 57.1% выборки):  
- Средний возраст: 42.88 года  
- Уровень образования: 10.36 лет  
- Среднее количество рабочих часов в неделю: 44.53  
- Заметно выше капитальный доход (1632 против 340)  
- Доля людей с доходом >50K: 36.3% (более чем в 4.5 раза выше, чем в кластере 0)

Также наблюдаются значимые различия по семейному статусу (в кластере 1 преобладают женатые/замужние) и полу (в кластере 1 значительно больше мужчин — 94% против 31%).  

Таким образом, кластеры чётко разделяют группы людей:  
- **Кластер 0** — молодые, менее образованные, часто одинокие люди с низким уровнем дохода.  
- **Кластер 1** — более образованные и взрослые люди с большей финансовой активностью.

### Вывод

Выполняя лабораторную работу, методом главных компонент были построены два новых признака, объясняющие 27.10% дисперсии исходных данных. На основе этих признаков выполнена кластеризация на 2 кластера.

Сравнительный анализ внутренних метрик качества показал, что кластеризация после применения PCA даёт результаты (0.1477), сопоставимые с KMeans из задания 2 (0.1494), но уступает агломеративной кластеризации (0.3067).

Полученные кластеры имеют чёткую интерпретацию и согласуются с реальными социально-экономическими закономерностями: высокий доход устойчиво связан с возрастом, уровнем образования, интенсивностью труда и наличием инвестиций.